# 🧵 Fabric Spot Defect Classifier
**EfficientNet-B0 Feature Extraction + SVM**

### Steps
1. Install dependencies
2. Mount Google Drive
3. Extract features with EfficientNet-B0 (frozen)
4. Train SVM classifier
5. Evaluate and save model

> ⚡ Make sure **Runtime → Change runtime type → T4 GPU** is selected before running.

## Cell 1 — Install Dependencies

In [ ]:
!pip install -q tqdm scikit-learn torch torchvision

## Cell 2 — Mount Google Drive

Upload your dataset to Google Drive under:
```
MyDrive/FabricSpotDefect/SeparatedDataset/<class_name>/<images>
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Cell 3 — Imports & Paths

In [ ]:
import os
import json
import numpy as np
import torch
import torch.nn as nn
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader
from pathlib import Path
from tqdm import tqdm
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.model_selection import train_test_split
import pickle

# ── Paths ──────────────────────────────────────────────────────────────
# Update BASE_DIR if your dataset is in a different Drive folder
BASE_DIR      = Path("/content/drive/MyDrive/FabricSpotDefect")
DATA_DIR      = BASE_DIR / "SeparatedDataset"
FEATURES_FILE = BASE_DIR / "features.npz"
MODEL_FILE    = BASE_DIR / "stain_classifier_best.pkl"
MAPPING_FILE  = BASE_DIR / "class_mapping.json"

BASE_DIR.mkdir(parents=True, exist_ok=True)
print(f"BASE_DIR: {BASE_DIR}")
print(f"DATA_DIR exists: {DATA_DIR.exists()}")

## Cell 4 — Feature Extractor (EfficientNet-B0)

In [ ]:
def get_feature_extractor(device):
    """Load EfficientNet-B0 with the classification head removed."""
    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
    model.classifier = nn.Identity()  # output: 1280-dim feature vector
    model = model.to(device)
    model.eval()
    return model


def extract_features(data_dir, device):
    """Extract 1280-dim feature vectors for all images using EfficientNet-B0."""
    transform = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    dataset = datasets.ImageFolder(data_dir, transform=transform)
    class_names = dataset.classes
    loader = DataLoader(
        dataset,
        batch_size=64,      # larger batch = faster on Colab GPU
        shuffle=False,
        num_workers=2,      # parallel data loading
        pin_memory=True     # faster CPU→GPU transfer
    )

    print(f"Classes: {class_names}")
    print(f"Total images: {len(dataset)}")

    model = get_feature_extractor(device)
    all_features, all_labels = [], []

    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Extracting features"):
            images = images.to(device, non_blocking=True)
            feats  = model(images)          # (batch, 1280)
            all_features.append(feats.cpu().numpy())
            all_labels.append(labels.numpy())

    all_features = np.concatenate(all_features, axis=0)
    all_labels   = np.concatenate(all_labels,   axis=0)
    return all_features, all_labels, class_names

print("Functions defined.")

## Cell 5 — Extract or Load Cached Features

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}\n")

if FEATURES_FILE.exists():
    print(f"Loading cached features from {FEATURES_FILE}...")
    data = np.load(FEATURES_FILE, allow_pickle=True)
    X = data["features"]
    y = data["labels"]
    class_names = list(data["class_names"])
    print(f"Loaded {len(X)} feature vectors.")
else:
    print("Extracting features with EfficientNet-B0 (frozen) — this takes ~5 min...")
    X, y, class_names = extract_features(DATA_DIR, device)
    np.savez(FEATURES_FILE, features=X, labels=y, class_names=class_names)
    print(f"\nSaved features to {FEATURES_FILE}")

# Save class mapping
with open(MAPPING_FILE, "w") as f:
    json.dump(class_names, f)

print(f"Classes: {class_names}")
print(f"Feature matrix shape: {X.shape}")

## Cell 6 — Train / Val / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.125, random_state=42, stratify=y_train
)
print(f"Split → Train: {len(X_train)}  Val: {len(X_val)}  Test: {len(X_test)}")

## Cell 7 — Train SVM Classifier

In [ ]:
print("Training Support Vector Machine (SVM) classifier...")
clf = SVC(
    kernel='rbf',
    C=10.0,
    gamma='scale',
    probability=True,
    random_state=42,
    verbose=True
)
clf.fit(X_train, y_train)
print("Training complete!")

## Cell 8 — Evaluate

In [ ]:
val_preds = clf.predict(X_val)
val_acc   = accuracy_score(y_val, val_preds)
print(f"Validation Accuracy: {val_acc*100:.2f}%")

test_preds = clf.predict(X_test)
test_acc   = accuracy_score(y_test, test_preds)
print(f"Test Accuracy:       {test_acc*100:.2f}%")

print("\n--- Classification Report (Test Set) ---")
print(classification_report(y_test, test_preds, target_names=class_names))

print("--- Confusion Matrix ---")
cm = confusion_matrix(y_test, test_preds)
print(f"       {class_names}")
for i, row in enumerate(cm):
    print(f"  {class_names[i][:10]:12s} | {row}")

## Cell 9 — Save Classifier to Drive

In [ ]:
with open(MODEL_FILE, "wb") as f:
    pickle.dump(clf, f)
print(f"Classifier saved to: {MODEL_FILE}")
print("Done! Use stain_classifier_best.pkl with predict.py for inference.")